In [14]:
import torch
import torch.nn as nn

class CircularSlidingWindowMean(nn.Module):
    def __init__(self, window_size=8):
        super(CircularSlidingWindowMean, self).__init__()
        self.window_size = window_size

    def forward(self, x):
        padding = self.window_size - 1
        x_padded = torch.cat([x, x[:, :padding, :]], dim=1)
        
        unfolded = x_padded.unfold(dimension=1, size=self.window_size, step=1)
        windowed_mean = unfolded.mean(dim=-1)
        
        return windowed_mean

# Example usage
batch_size = 256
num_points = 32
num_features = 3

# Create random input tensor
x = torch.ones(batch_size, num_points, num_features)
x[:, -1, :] = 0

# Apply circular sliding window mean
layer = CircularSlidingWindowMean(window_size=8)
output = layer(x)
print(x)
print(output)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

tensor([[[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [0., 0., 0.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [0., 0., 0.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [0., 0., 0.]],

        ...,

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [0., 0., 0.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [0., 0., 0.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [0., 0., 0.]]])
tensor([[[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1

In [ ]:
class CircularSlidingWindowWeightedMean(nn.Module):
    def __init__(self, window_size=8):
        super(CircularSlidingWindowWeightedMean, self).__init__()
        self.window_size = window_size
        
        # Create decreasing weights
        weights = torch.linspace(1, 0, steps=window_size, endpoint=False)
        weights = weights.flip(0)  # Reverse the order to have decreasing weights
        self.register_buffer('weights', weights.unsqueeze(0).unsqueeze(2))

    def forward(self, x):
        padding = self.window_size - 1
        x_padded = torch.cat([x, x[:, :padding, :]], dim=1)
       
        unfolded = x_padded.unfold(dimension=1, size=self.window_size, step=1)
        
        # Apply weights to the unfolded tensor
        weighted_unfolded = unfolded * self.weights
        
        # Compute weighted mean
        windowed_weighted_mean = weighted_unfolded.sum(dim=-1) / self.weights.sum()
       
        return windowed_weighted_mean